# Task

With AI-modelling and Data Science there is plenty of opportunity to improve processes or suggest improved ways of doing things. When doing so it is often very smart and efficient (time is a scarce resource) to create a POC (Proof of Concept) which basically is a small demo checking wether it is worthwile going further with something. It is also something concrete which facilitates discussions, do not underestimate the power of that. 

In this example, you are working in a company that sells cars and they have a "manual" process of setting prices by humans. You want to understand better what affects the car prices and you think you can make this process better by using Machine Learning. Your task is to create a POC that you will present to your team colleagues and use as a source of discussion of wether or not you should continue with more detailed modelling. 

Two quotes to facilitate your reflection on the value of creating a PoC: 

"*Premature optimization is the root of all evil*". 

"*Fail fast*".


**More specifially, do the following:**
1. Split your data into `X` and `y`, and then into train, validation and test sets.
2. An EDA (Exploratory Data Analysis) of the cars data set. This is your main task. 
3. Preparing the data for modelling. 
4. Create one `LinearRegression` model that can predict car prices.
Use root mean squared error (RMSE) as a metric to evaluate your model.
5.  Evaluate your model on the test set using RMSE as the metric. 
What are your conclusions?
What could be the next step? Is the POC convincing enough or is it not worthwile continuing? Do we need to dig deeper into this before taking some decisions? 


In [ ]:
# Import libraries

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

Scikit-learn är ett mycket populärt tredjepartsbibliotek i Python för maskininlärning som används för dataanalys och modellering. Det innehåller färdiga verktyg för t.ex. klassificering, regression, klustring, dimensionsreduktion och modellutvärdering, och bygger på bland annat NumPy och matplotlib. Vi kommer under kursens gång lära oss om alla dessa koncept och om biblioteket scikit-learn. Se detta endast som ett smakprov, INTE något som ni ska förstå i detalj. 

### Load data

In [ ]:
# Load Excel file to pandas
# Below, set your own path where you have stored the data file if it is not in the same folder as the Juptyer Notebook file. 
cars = pd.read_csv("car_price_dataset.csv", sep=";") 

In [ ]:
cars.info()

In [ ]:
# Split your data into `X` and `y`, and then into train, validation and test sets.

X = cars.drop(columns=["Price"])
y = cars["Price"]

# Train (60%) Temp (40%)
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.4, random_state=42)

# Validation (20%) Test (20%)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

### EDA
To start the off the POC, it is essential to understand the key factors that shape the car market.
This analysis should examine the primary influences on car pricing, including brand, model, year, engine size, fuel type, transmission, mileage, and ownership history.

By identifying trends and correlations, the goal is to make informed, data-driven decisions that optimize inventory management, pricing strategies, and market positioning.

In [ ]:
# EDA should only be performed on the training dataset to avoid data leakage.

train_cars = X_train.copy()
train_cars["Price"] = y_train

train_cars.info()

### Preparing data

In [ ]:
# Remove rows with missing values from the table. (not sure you have any but you can assume so)
X_train.dropna(inplace=True)
X_val.dropna(inplace=True)
X_test.dropna(inplace=True)

print(X_train.isnull().sum().sum())
print(X_val.isnull().sum().sum())
print(X_test.isnull().sum().sum())

In [ ]:
# Drop categorical columns if they seem to have a low impact on the price, then you only have numeric columns which will simplify your analysis. Remember, this is a POC!
# Use dummy-variable-encoding for categorical columns you wish to keep. Makes categorical variables numerical. 

X_train.drop(columns=["Model"], inplace=True)
print(X_train.info())
X_val.drop(columns=["Model"], inplace=True)
X_test.drop(columns=["Model"], inplace=True)

encoder = OneHotEncoder(sparse_output=False, drop="first")

X_train = encoder.fit_transform(X_train[["Brand", "Fuel_Type", "Transmission"]])
X_val = encoder.transform(X_val[["Brand", "Fuel_Type", "Transmission"]])
X_test = encoder.transform(X_test[["Brand", "Fuel_Type", "Transmission"]])

In [ ]:
X_train_encoded = pd.DataFrame(
    X_train,
    columns=encoder.get_feature_names_out(["Brand", "Fuel_Type", "Transmission"])
)

X_train_encoded.head()


In [ ]:
# Standardization
# Ensure all features are on the same scale 

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

### Model

In [ ]:
# Linear Regression 

# Train a linear regression model on the training data

linreg_model = LinearRegression()
linreg_model.fit(X_train, y_train)

# Evaluate the model on the validation data
y_pred_linreg = linreg_model.predict(X_val)

rmse_linreg = np.sqrt(mean_squared_error(y_val, y_pred_linreg))
print(f"RMSE Linear Regression: {rmse_linreg}")

### Evaluation

In [ ]:
# Evaluate the choosen model on the test data

# Train + Validation
X_train_val = np.vstack((X_train, X_val))
y_train_val = np.hstack((y_train, y_val))

# Retrain linear regression model on training and validation data
linreg_model_final = LinearRegression()
linreg_model_final.fit(X_train_val, y_train_val)

y_test_pred = linreg_model_final.predict(X_test)

rmse_test = np.sqrt(mean_squared_error(y_test, y_test_pred))
print(f"Final RMSE Linear Regression: {rmse_test:.4f}")

### Conclusions

RMSE anger hur långt ifrån det sanna priset modellens prediktioner ligger i genomsnitt, mätt i samma enhet som priset.

Har vi skapat en bra modell? 